### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [1]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [37]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- math and reasoning
- planning and organization

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very helpful, but I can also make mistakes, so important facts should be verified.
- I don’t automatically know real-time information unless it’s provided to me or I have access to tools that supply it.

If you want, I can also tell you about:
- my strengths and limitations
- how I “think” at a high level
- what I’m good at compared with search engines
- how to get better answers from me

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [40]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [13]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "LLM Leaderboard - Comparison of over 100 AI models from OpenAI, Google ...",
      "url": "https://artificialanalysis.ai/leaderboards/models",
      "description": "Comparison and ranking the performance of over 100 AI <b>models</b> (<b>LLMs</b>) across key metrics including intelligence, price, performance and speed (output speed - tokens per second &amp; laten

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools and error handling

In [18]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [ ]:
import httpx
from langchain.tools import tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [32]:
fetch_webpage_content("https://blog.cloudflare.com/agents-week-in-review/")

"  Building the agentic cloud: everything we launched during Agents Week 2026\n\n[Get Started Free](https://dash.cloudflare.com/sign-up)|[Contact Sales](https://www.cloudflare.com/plans/enterprise/contact/)|\n\n▼\n\n[![The Cloudflare Blog](https://cf-assets.www.cloudflare.com/zkvhlag99gkb/69RwBidpiEHCDZ9rFVVk7T/092507edbed698420b89658e5a6d5105/CF_logo_stacked_blktype.png)](/)\n\n[The Cloudflare Blog](/)\n------------------------\n\nSubscribe to receive notifications of new posts:\n\nSubscribe\n\n![magnifier icon](/images/magnifier.svg)![hamburger menu](/images/hamburger.svg)\n\n[AI](/tag/ai/)\n\n[Developers](/tag/developers/)\n\n[Radar](/tag/cloudflare-radar/)\n\n[Product News](/tag/product-news/)\n\n[Security](/tag/security/)\n\n[Policy & Legal](/tag/policy/)\n\n[Zero Trust](/tag/zero-trust/)\n\n[Speed & Reliability](/tag/speed-and-reliability/)\n\n[Life at Cloudflare](/tag/life-at-cloudflare/)\n\n[Partners](/tag/partners/)\n\n[AI](/tag/ai/)\n\n[Developers](/tag/developers/)\n\n[Radar

In [ ]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""
        What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
        Search the web for 50 result pages. Fetch webpages.
    """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


    What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    Search the web for 50 result pages. Fetch webpages.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_qTHfbsggmLq6l7W3QiJRNQio)
 Call ID: call_qTHfbsggmLq6l7W3QiJRNQio
  Args:
    query: latest news updates AI agents Microsoft Google AWS Anthropic OpenAI
    limit: 50
    searchMode: auto
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest news updates AI agents Microsoft Google AWS Anthropic OpenAI",\n  "engines": [\n    "startpage",\n    "duckduckgo"\n  ],\n  "totalResults": 50,\n  "results": [\n    {\n      "title": "Microsoft\'s multi-agent AI system tops Anthropic\'s Mythos ... - GeekWire",\n      "

I searched the web for **50 result pages** and fetched a representative set of webpages, including official/vendor sources and reporting. Some pages were blocked or rate-limited during fetches.

## What I found: latest AI agent updates by company

### Microsoft
**Key update:** Microsoft and OpenAI restructured their partnership.
- Microsoft says OpenAI remains its **primary cloud partner**, and OpenAI products ship first on Azure unless Microsoft can’t support required capabilities.
- OpenAI can now serve products across **any cloud provider**.
- Microsoft’s OpenAI IP license continues through **2032**, but is now **non-exclusive**.
- Microsoft no longer pays revenue share to OpenAI; OpenAI revenue share to Microsoft continues through **2030** with a cap.  
Source fetched: Microsoft official blog, Apr. 27, 2026.

**Agent/security angle:**
- Microsoft is participating in Anthropic’s **Project Glasswing**, using Claude Mythos Preview for defensive cybersecurity work.
- Anthropic quotes Microsoft EVP Igor Tsyganskiy saying Mythos Preview showed substantial improvements on Microsoft’s CTI-REALM benchmark.  
Source fetched: Anthropic Project Glasswing.

**Notable market signal:**
- CRN reports a senior Microsoft Copilot Security/AI leader, **Shawn Bice**, moved to AWS to help lead agentic AI and automated reasoning.  
Source fetched: CRN.

---

### Google
**Key update:** At **Google Cloud Next ’26**, Google made a major push around enterprise agents.
- Announced **Gemini Enterprise Agent Platform**
- Positioned it as a platform to **build, scale, govern, and optimize agents**
- Highlighted “agentic enterprise” momentum and rising token usage
- Announced supporting infrastructure including **8th-gen TPUs** and broader cloud stack updates  
Source fetched: Google official Keyword / Cloud Next ’26 collection.

**Additional reporting on agent features:**
Bloomberg/Mercury News says Google introduced:
- tools to **build AI agents**
- tracking/monitoring for agent work in companies
- **Memory Bank** and **Memory Profile**
- **Agent Simulation** for testing before launch
- Workspace-centered agent experiences, including no-code agent creation and collaboration features
- cybersecurity agents  
Source fetched: Mercury News / Bloomberg, Apr. 22, 2026.

**Anthropic relationship:**
- Google is also a partner in **Project Glasswing**
- Anthropic says Mythos Preview is available to participants via **Vertex AI**  
Source fetched: Anthropic Project Glasswing.

---

### AWS
**Key update:** AWS continues expanding its agent stack with **Amazon Bedrock Agents**.
Official AWS page highlights:
- **multi-agent collaboration**
- RAG integration with company data
- orchestration/execution across APIs and systems
- **memory retention**
- **code interpretation**
- also introduced **Amazon Bedrock AgentCore** for deploying and operating agents securely at scale using open-source frameworks and models  
Source fetched: AWS Bedrock Agents official page.

**OpenAI angle on AWS:**
- AWS Bedrock page explicitly links to **“Bedrock Managed Agents, powered by OpenAI”**, signaling deeper OpenAI presence on AWS.
- The New Stack reports the Microsoft-OpenAI restructure opened the door for OpenAI to work more broadly with **AWS and Google Cloud**.  
Sources fetched: AWS official page; The New Stack.

**Leadership / reliability focus:**
- AWS hired ex-Microsoft VP **Shawn Bice** to lead automated reasoning / neurosymbolic AI efforts for more reliable and trustworthy agents.  
Source fetched: CRN.

**Anthropic relationship:**
- Anthropic newsroom says **Anthropic and Amazon expanded collaboration** for up to **5 gigawatts of new compute**.
- AWS is also a launch partner in **Project Glasswing**.  
Sources fetched: Anthropic newsroom; Project Glasswing.

---

### Anthropic
**Key updates from Anthropic newsroom:**
- **Claude Opus 4.7** launched, with stronger performance across **coding, agents, vision, and multi-step tasks**
- **Claude Design** launched from Anthropic Labs
- **Project Glasswing** announced
- New vertical/enterprise pushes:
  - **Agents for financial services**
  - **Claude for Small Business**
  - **Claude for Creative Work**
- Anthropic and Amazon expanded collaboration for up to **5 GW of compute**  
Source fetched: Anthropic newsroom.

**Most important agent/security development:** **Project Glasswing**
- Anthropic says **Claude Mythos Preview** is a highly capable unreleased frontier model with strong cyber and agentic coding abilities.
- Claims it found thousands of high-severity vulnerabilities, including in major OSes and browsers.
- Partners include **AWS, Google, Microsoft, Apple, Cisco, CrowdStrike, JPMorganChase, NVIDIA, Palo Alto Networks**, and others.
- Anthropic says Project Glasswing participants can access Mythos Preview via:
  - Claude API
  - **Amazon Bedrock**
  - **Google Vertex AI**
  - **Microsoft Foundry**  
Source fetched: Anthropic Project Glasswing.

**Model benchmarks highlighted by Anthropic:**
- Strong gains on SWE-bench, Terminal-Bench, BrowseComp, OSWorld-Verified, etc.
- Anthropic frames Mythos’s cyber capability as driven by strong **agentic coding and reasoning**.  
Source fetched: Anthropic Project Glasswing.

---

### OpenAI
**Most concrete update in fetched sources:** OpenAI’s partnership shift with Microsoft.
- OpenAI can now distribute across **multiple clouds**
- Azure remains first/primary in many cases, but exclusivity has loosened materially  
Source fetched: Microsoft official post; The New Stack analysis.

**AWS distribution / agents signal:**
- AWS now references **Bedrock Managed Agents, powered by OpenAI**
- The New Stack says the reset enables OpenAI to reach enterprise customers that prefer **Bedrock** or **Vertex** inside existing security perimeters  
Sources fetched: AWS official page; The New Stack.

**What I could not directly fetch:**
- OpenAI’s own newsroom/news page returned **403 Forbidden**
- So I could not directly verify latest OpenAI agent announcements from OpenAI’s site in this run

---

## Cross-company themes
1. **Multi-cloud agent distribution is accelerating**
   - OpenAI is no longer effectively Azure-only
   - Anthropic is across AWS, Google, and Microsoft platforms
   - Enterprises want models inside existing cloud/security boundaries

2. **Agent platforms are maturing**
   - Google: Gemini Enterprise Agent Platform
   - AWS: Bedrock Agents + AgentCore
   - Anthropic: broader vertical agent solutions
   - Microsoft: increasingly flexible model ecosystem beyond only OpenAI

3. **Reliability, governance, and security are the next battleground**
   - Agent testing, memory, supervision, and evaluation are being productized
   - Cybersecurity-agent use cases are now central, especially via Anthropic’s Glasswing

4. **Coding agents remain a core competitive front**
   - Google is trying to catch OpenAI and Anthropic in coding
   - Anthropic is emphasizing agentic coding strength
   - AWS and Microsoft are both stressing trustworthiness and automated reasoning

---

## Fetch results summary
### Successfully fetched key pages
- Microsoft official blog: Microsoft-OpenAI partnership update
- Google official Cloud Next ’26 page
- Anthropic newsroom
- Anthropic Project Glasswing
- AWS Bedrock Agents official page
- Mercury News/Bloomberg on Google agents
- The New Stack on Microsoft/OpenAI restructure
- CRN on AWS hiring Microsoft AI/security exec

### Fetch failures / blocked pages
- Reuters AI page: **401**
- VentureBeat article: **429**
- GeekWire article: **403**
- OpenAI news page: **403**

---

## Concise answer
If you want the **latest meaningful AI agent updates** from these companies based on the fetched pages:

- **Microsoft:** loosened OpenAI exclusivity; now more flexibility across clouds and models.
- **Google:** launched **Gemini Enterprise Agent Platform** and multiple enterprise agent-building/testing/memory tools at Cloud Next.
- **AWS:** expanding **Bedrock Agents**, **multi-agent collaboration**, **memory**, **code interpretation**, and **AgentCore**; also surfacing OpenAI-powered managed agents.
- **Anthropic:** pushing hard on enterprise and security agents; **Project Glasswing** is the biggest new agent/security announcement; also launched **Claude Opus 4.7**.
- **OpenAI:** biggest verified update here is broader **multi-cloud distribution** after the Microsoft partnership reset; direct OpenAI newsroom fetch was blocked.

If you want, I can next turn this into a **clean table with columns: company / announcement / date / agent relevance / source URL**.